# Assignment 2 — Georgian Spellchecker: Data and Training

This notebook builds a character-level sequence-to-sequence spellchecker for Georgian words.

The model learns mappings like:

- `გამარჰონა → გამარჯობა`
- `პროგამა → პროგრამა`
- `თბილისი → თბილისი`

It uses a recurrent encoder-decoder GRU model and works at character level only.


In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
import torch

# Keep CPU training predictable and fast on normal laptops / Colab CPU.
torch.set_num_threads(1)

from spellchecker import (
    load_words, corrupt_word, make_pairs, build_vocab,
    train_model, correct_word, download_words_from_url
)


## 1. Data collection

The assignment requires a source of correctly spelled Georgian words. In this project, the word list is loaded from a Google Drive text file:

`https://drive.google.com/file/d/1NnELSMHpI9ru6RgzGVT6wiIiTAjWKgj_/view?usp=sharing`

The file is also cached locally as:

`data/georgian_words.txt`

This means the project can run offline, but when internet is available the notebook can download the latest version from Drive. If the Drive file changes, rerunning this notebook refreshes the local dataset and retrains the model on the updated data.

The preprocessing rule is to keep only Georgian-script words and remove duplicates.


In [ ]:
DATASET_URL = 'https://drive.google.com/file/d/1NnELSMHpI9ru6RgzGVT6wiIiTAjWKgj_/view?usp=sharing'
words_path = 'data/georgian_words.txt'

# Try to refresh the local dataset from Google Drive.
# If the notebook is run without internet, it falls back to the cached local file.
try:
    download_words_from_url(DATASET_URL, words_path)
    print('Dataset refreshed from Google Drive.')
except Exception as e:
    print('Could not download from Drive; using cached local dataset instead.')
    print(type(e).__name__, e)

words = load_words(words_path)
print('Unique words:', len(words))
print(words[:30])


## 2. Data generation

The model needs `(input_word, target_word)` pairs. The target is always the correct word.

I generate realistic synthetic typos using four common edit operations:

1. character deletion
2. character replacement
3. adjacent character swap
4. extra character insertion

I also include already-correct inputs, because the model must learn not to change words that are already correct.

In [ ]:
random.seed(7)
for w in ['გამარჯობა', 'თბილისი', 'პროგრამა', 'საქართველო', 'კომპიუტერი']:
    print(w, '->', [corrupt_word(w, p_keep_correct=0.0) for _ in range(5)])

In [ ]:
pairs = make_pairs(words[:1000], variants_per_word=3)
print('Example training pairs:')
for pair in pairs[:20]:
    print(pair[0], '→', pair[1])

## 3. Model design

I use an encoder-decoder GRU model. This is a good fit because the misspelled input and corrected output can have different lengths.

For example:

`პროგამა → პროგრამა`

The output is longer than the input because the missing `რ` must be inserted.

Special tokens:

- `<PAD>` for padding batches
- `<SOS>` for decoder start
- `<EOS>` for end of word
- `<UNK>` for unknown characters

In [ ]:
stoi = build_vocab(words)
print('Vocabulary size:', len(stoi))
print(stoi)

## 4. Training

The training function performs:

- train/validation split
- batching and padding
- teacher forcing with decoder input
- cross-entropy loss ignoring `<PAD>`
- tqdm progress bar for each epoch
- checkpoint saving when validation loss improves

This run uses **30 epochs** so the loss curve has enough points for the assignment report.


In [ ]:
model, stoi, history = train_model(
    words_path=words_path,
    model_path='model/georgian_spellchecker.pt',
    epochs=30,
    batch_size=32,
    lr=0.001,
    max_words=2800
)


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

## 5. Save

The trained model is saved to:

`model/georgian_spellchecker.pt`

In [ ]:
print(Path('model/georgian_spellchecker.pt').exists())